<h1>Week 7: Penetration Testing</h1>

<h2>Section 1: WHOis Domain lookup</h2>

In [4]:
import socket
import requests

def get_domain_info(domain):
    try:
        # Get IP address (active but low-risk)
        ip = socket.gethostbyname(domain)
        print(f"IP Address: {ip}")
        # Get public WHOIS-like info (passive, using a free API)
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(f"https://ipapi.co/{ip}/json/", headers=headers)
        if response.status_code == 200:
            print(response.status_code)
            data = response.json()
            print(f"Organization: {data.get('org', 'Unknown')}")
            print(f"City: {data.get('city', 'Unknown')}")
            print(f"Country: {data.get('country_name', 'Unknown')}")
        else:
            print("Could not fetch WHOIS data.")
    except Exception as e:
        print(f"Error: {e}")
    # Example: Use a public domain (never use without permission!)
get_domain_info("python.com")

IP Address: 185.158.133.1
200
Organization: CLOUDFLARENET
City: Frankfurt am Main
Country: Germany


<h2>Section 2: Simulating black box on a web page</h2>

In [ ]:
import requests

# does black bost reconnaisance step by sending a http head to target url
def black_box_recon(url):
    try:
        response = requests.head(url)
        # prints out the results
        print(response)
        print("Black Box Findings:")
        print(f"Server: {response.headers.get('Server', 'Unknown')}")
        print(f"Content-Type: {response.headers.get('Content-Type',
        'Unknown')}")
        print(response.headers)
    except Exception as e:
        print(f"Error: {e}")
url = "https://python.com"
known_info = {"server": "Apache 2.4", "vulns": "Check CVE-2021-1234"}
black_box_recon(url)

<Response [200]>
Black Box Findings:
Server: cloudflare
Content-Type: text/html; charset=utf-8
{'Date': 'Wed, 19 Nov 2025 12:27:27 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Connection': 'keep-alive', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'X-Content-Type-Options': 'nosniff', 'Vary': 'accept-encoding', 'ETag': 'W/"2e0e52860ed68af6a8136e677cfe4e77"', 'Content-Encoding': 'gzip', 'Server': 'cloudflare', 'CF-RAY': '9a0fb3a78e3fada3-LHR'}


<h2>Section 3: Basic Port scanning using pure python</h2>

In [ ]:
import socket
def scan_ports(host, ports):
    open_ports = []
    for port in ports:
        #  creates tcp socket
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        result = sock.connect_ex((host, port))
        if result == 0:
            open_ports.append(port)
        sock.close()
    return open_ports
# //has target host and ip and scans for those ports
host = "10.0.0.7"
ports = [80, 443, 22, 8080]
open_ports = scan_ports(host, ports)
print(f"Open ports on {host}: {open_ports}")

Open ports on 10.0.0.7: [8080]


<h2>Section 4: Using Nmap in python</h2>

In [8]:
import nmap

def nmap_scan(host, port_range='1-1024'):
    nm = nmap.PortScanner()
    try:
        nm.scan(host, port_range, arguments='-sV') # -sV for service version detection
        for host in nm.all_hosts():
            print(f"Host: {host} ({nm[host].hostname()})")
            print(f"State: {nm[host].state()}")
            for proto in nm[host].all_protocols():
                print(f"Protocol: {proto}")
                lport = nm[host][proto].keys()
                for port in sorted(lport):
                    service = nm[host][proto][port]
                    print(f"Port: {port}\tState: {service['state']}\tService: {service.get('name', 'unknown')} {service.get('version', '')}")
    except Exception as e:
        print(f"Error: {e}")

# Example: Scan localhost
nmap_scan('127.0.0.1', '1-10')

Host: 127.0.0.1 (localhost)
State: up
Protocol: tcp
Port: 1	State: closed	Service: tcpmux 
Port: 2	State: closed	Service: compressnet 
Port: 3	State: closed	Service: compressnet 
Port: 4	State: closed	Service:  
Port: 5	State: closed	Service: rje 
Port: 6	State: closed	Service:  
Port: 7	State: closed	Service: echo 
Port: 8	State: closed	Service:  
Port: 9	State: closed	Service: discard 
Port: 10	State: closed	Service:  


<h2> (Challenge) Section 5: Scanning a virtual machine</h2>

In [ ]:
import nmap

def nmap_scan_with_os(host, port_range='1-1024'):
    nm = nmap.PortScanner() #creates nmap scanner object
    try:
        #runs the nmap service with os destection enabled
        nm.scan(host, port_range, arguments='-sV -O')

        print(f"Host: {host}")
        print(f"State: {nm[host].state()}")

        if 'osmatch' in nm[host]:
            print("\nOperating Systems:")
            for os in nm[host]['osmatch']:
                #prints the OS match name and accuracy score
                print(f" - {os['name']} (accuracy: {os['accuracy']}%)")
        else:
            print("\nno OS information found ")
        # Loops through detected ports for the protocol
        for proto in nm[host].all_protocols():
            print(f"\nProtocol: {proto}")
            #displays the port state, service name and version
            for port in sorted(nm[host][proto].keys()):
                service = nm[host][proto][port]
                print(f"Port: {port}\tState: {service['state']}\tService: {service.get('name', '')} {service.get('version', '')}")

    except Exception as e:
        print(f"Error: {e}")

nmap_scan_with_os('192.168.56.101', '1-1024')


Host: 192.168.1.10
State: up

Operating Systems:
 - Linux 4.15 - 5.19 (accuracy: 100%)
 - MikroTik RouterOS 7.2 - 7.5 (Linux 5.6.3) (accuracy: 100%)

Protocol: tcp
Port: 22	State: open	Service: ssh 
Port: 53	State: open	Service: domain 
Port: 80	State: open	Service: http 
Port: 443	State: open	Service: http 


<h2>(Challenge) Section 6: Finding live hosts that are up</h2>

In [ ]:
import nmap
import ipaddress
#Performs a basic discovery scan across a subnet to identify devices on existing network
def discover_hosts(network):
    nm = nmap.PortScanner()
    print(f"\nDiscovering devices on: {network}\n")

    try:
        #converts network object to a string
        nm.scan(hosts=str(network), arguments='-sn')
        #Identifies all hosts that responded with up state and than prints out those hosts.
        live_hosts = [host for host in nm.all_hosts() if nm[host].state() == 'up']
        print(f"Devices found: {len(live_hosts)}\n")
        for host in live_hosts:
            print(f" {host}")
        return live_hosts

    except Exception as e:
        print(f"Error discovering network: {e}")
        return []


if __name__ == "__main__":
    network = ipaddress.ip_network("192.168.1.0/24", strict=False)
    
    discover_hosts(network)



Discovering devices on: 192.168.1.0/24

Devices found: 8

 192.168.1.10
 192.168.1.100
 192.168.1.20
 192.168.1.254
 192.168.1.69
 192.168.1.70
 192.168.1.78
 192.168.1.85
